In [1]:
# Essential imports
import re

from io import StringIO

import pandas as pd
import numpy as np

from Bio import SeqIO
from bioservices import UniProt

import tqdm

# Connect to UniProt service
u = UniProt(verbose=False)

In [4]:
my_file = open("Accession.txt", "r") 
  
# reading the file 
data = my_file.read() 
  
# replacing end splitting the text  
# when newline ('\n') is seen. 
data_into_list = data.split("\n") 
print(data_into_list) 
my_file.close() 

['A0A011VQM7', 'A0A063ZJU6', 'A0A081EVR3', 'A0A099I9L1', 'A0A0D8IWY3', 'A0A0E2HBL2', 'A0A0F0CIE1', 'A0A0F9HQB3', 'A0A0G3WG46', 'A0A0G9LEF6', 'A0A0H5SG89', 'A0A0J6WZA7', 'A0A0J6ZRU4', 'A0A0J9EMW5', 'A0A0M2NHW4', 'A0A0M6WC26', 'A0A0M6WCZ2', 'A0A0M9AS13', 'A0A0R2HL96', 'A0A0S2W112', 'A0A0U5JBG5', 'A0A0W7TMH6', 'A0A0X8VAS2', 'A0A136Q6K1', 'A0A136WGT4', 'A0A140DTE7', 'A0A143Y517', 'A0A143Y9S4', 'A0A143YNF2', 'A0A143YTA3', 'A0A143YWY4', 'A0A143YZV1', 'A0A143Z212', 'A0A143ZCA8', 'A0A151AI36', 'A0A169X118', 'A0A173R3X4', 'A0A173R662', 'A0A173SMJ4', 'A0A173TUR6', 'A0A173V441', 'A0A173W538', 'A0A173X4T2', 'A0A173XV88', 'A0A173XVK8', 'A0A173Z4K3', 'A0A173ZTK6', 'A0A174C5U1', 'A0A174D1D6', 'A0A174GH26', 'A0A174H6C4', 'A0A174JG28', 'A0A174RYE8', 'A0A174T4F8', 'A0A174TIU3', 'A0A174VZ55', 'A0A174XKK3', 'A0A174YTX7', 'A0A174Z5H4', 'A0A175A4T9', 'A0A175ABC8', 'A0A176U687', 'A0A1B1YE78', 'A0A1B1YLE9', 'A0A1C0BVB4', 'A0A1C5KIM4', 'A0A1C5M0W8', 'A0A1C5MDL8', 'A0A1C5MK12', 'A0A1C5N7S1', 'A0A1C5PFK7', 'A0A1

TH: First I search with one ID and take the result and put it into a dataframe. I'm not sure how to get the column of the data frame out so that I can search with extract seq later on but I guess I need to just save it as a string?

LP: The format here looks a little like EMBL format, which in adapted form is the GenBank flat file format we still use (.gbff). It's not quite the same, but the "trick" is - as ever - to work out what information is there that you need, and then decide how much effort you want to spend parsing it. We could, if we wanted, create a new data type or data class, and parse the data into that (worth it if you're doing this a lot and need reusability), or we could simply parse into a collection of (start, end) tuples if we're writing one-shot or throwaway code. Either way, you've gone a long way towards the parsing in your code below.

LP: I'll go through the approach I would take in parsing out the start/end co-ordinates for each helix, below.

In [5]:
# Putting this all into a function:
def parse_ft_transmembrane(result):
    """Returns start, end positions from ft_transmembrane result as generator"""
    transmems = result.split("\n")[1]
    positions = re.findall("(?<=TRANSMEM )(?P<tmemstart>[0-9]*)..(?P<tmemend>[0-9]*)", transmems)
    return ((int(start), int(end)) for start, end in positions)

LP: So much for parsing out the start and end points. That's only part of what you want to do, though. You also want the corresponding protein sequence. We can get this direct from UniProt for the same query in FASTA format.

LP: We need to parse this, and Biopython is convenient. But Biopython needs input as a file-like object, so we use `StringIO` to convert our FASTA string, and make a Biopython `SeqRecord` object.

LP: Bringing it all together, we can extract helix sequences as below. We need to be careful with off-by-one numbering, as Biopython sequence are 0-indexed, but UniProt sequences are 1-indexed.

```text
Biopython: 012345678
Sequence:  MCMFCLEPP
Uniprot:   123456789
```

LP: Let's be neat and wrap this all in a function, where we just pass the query ID.

In [6]:
# Function to return TM helix sequences from UniProt, given a query AA ID
def get_tm_helices(query):
    """Return generator of TM helix sequences for the passed UniProt query"""
    tms = u.search(query, frmt="tsv", columns='ft_transmem')
    record = SeqIO.read(StringIO(u.search(query, frmt="fasta")), "fasta")
    # for returns in record:
    #     if returns != 1:
    #        continue
    return (record.seq[start-1:end] for start, end in parse_ft_transmembrane(tms))

LP: Let's try this on the original list of queries

In [7]:
# returning IDs and TM sequences of proteins with 11 or 12 predicted helices which
# matches ammonium transporters Amt/Mep/Rh

lest = []

spoen = []

matches = []

for quas in data_into_list:
    print(f"\nQuery ID: {quas}")
    helixe = list(get_tm_helices(quas))
    if len(helixe) == 12:
        print(f'query {quas} has {len(helixe)}, using last 11 helices')
        helixe = helixe[-11:]
    if len(helixe) < 11:
        print(f'query {quas} has {len(helixe)} which is too few, skipping')
        helixe = ''
    if len(helixe) > 12:
        print(f'query {quas} has {len(helixe)} which is too many, skipping')
        helixe = ''
    for idx, helix in enumerate(helixe):     
        print(f"Helix {idx + 1}: {helix}")
        reshault = f">{quas}_Helix_{idx + 1} {helix}"
        staup = quas
        lest.append(reshault)
        spoen.append(staup)
        spoon = f"{quas}_Helix_{idx + 1}"
        matches.append(spoon)


Query ID: A0A011VQM7
Helix 1: GVWFMIAVALVFFMQAGFAMV
Helix 2: FCIGTVAFIAIGFGLLLG
Helix 3: SFAWDQFVFNLVFCATTATIVS
Helix 4: YCIYSGIISALIYPIEAHWIWG
Helix 5: LSGSCAIHMVGGICALIGAKIVG
Helix 6: GHNLIIGALGVFILWFGWYG
Helix 7: IFLTTTVAPAIATVTVMVFTWL
Helix 8: VSMCLNASLAGLVGITAGC
Helix 9: FGAIMVGIVSGFLVCFGVWFL
Helix 10: AVAVHMMNGIWGTIAVGLFA
Helix 11: LGLQLLGFVSVAAWAAVAITITFLII

Query ID: A0A063ZJU6
Helix 1: MWVLIATFLIFFMHAGFAML
Helix 2: WSVGVIAFFFVGFGVNALTGSVT
Helix 3: WVSWLFSAVFAMTAATIVSGAVAG
Helix 4: LLLAAIIYPVVTGFTWGGGLLAI
Helix 5: FAGGMIVHGMGGIAGLTAAWII
Helix 6: LTFAALGTLILAFGWYGF
Helix 7: VALNTTLGMAAGAIGAGAVALY
Helix 8: TLYVANGLLAGLVGVTASA
Helix 9: YGGLLSGLIAGAQLPIVFSLV
Helix 10: VFPVHGSAGVIGALLLPFFHI
Helix 11: IVGVTVIGVWTVATTAIVFGAF

Query ID: A0A081EVR3
Helix 1: LMWVAVVCFLIFFMHAGF
Helix 2: LLTWGVGVLVYFLVGFGIEGVA
Helix 3: WLFGAVFAMTAATIVSGAVAG
Helix 4: YVAYTIAISAVIYPVVAGITWGGG
Helix 5: FAGGMIVHGVGGIAGLTAAYML
Helix 6: MTFAVLGTLVLAFGWYGF
Helix 7: GGYSAVGRVSLTTTLGMGAGAIGAAL
Helix 8: LYVANGVLAGL

ValueError: No records found in handle

query A0A511NGC7 has 12, using last 11 helices
Helix 1: VAWILAAAGLVLLMTPGLSFFYG
Helix 2: MLQSFIALGVISILWVVVGFSLSF
Helix 3: IPFVLFALFQMKFAVITPALVT
Helix 4: YLLFMILFSLIIYTPLCHMVW
Helix 5: VVHMSAGFAALAGALVIG
Helix 6: IPFVMLGTGMLWFGWFGF
Helix 7: AALAFGTTTIASASGMMTWILF
Helix 8: SVSAMGACIGAVVGLVAI
Helix 9: SIFFGFITAIISNMAMYW
Helix 10: VFACHGVGGIMGMILTAIFA
Helix 11: FIHHIIALFFVSVFTFFGAITLFYL

Query ID: A0A142BI19
Helix 1: FYFLMSGALVMWMAAGFAMLEAGLV
Helix 2: NIALFAIACTMYLLCGYAIM
Helix 3: FFFQVVFVATAMSIVSGAVA
Helix 4: LWSFLLFAVFMTGFIYPV
Helix 5: AGSGVVHMAGAAAALAGVLLL
Helix 6: LPLATLGTFILWLGWFGF
Helix 7: IFVNTNAGAAGGVIAALIVA
Helix 8: ADLTMALNGALAGLVAITA
Helix 9: SALAATLIGAVGGSLVVFAIVAL
Helix 10: ISVHGVVGIWGLLAVPLT
Helix 11: LSAQLLGIVSIFAWVFGASLVVWGVV

Query ID: A0A081N2G5
Helix 1: FYFLMSGALVMWMAAGFAMLEAGLV
Helix 2: NIALFAIACTMYLLCGYAIM
Helix 3: FFFQVVFVATAMSIVSGAVA
Helix 4: LWSFLLFAVFMTGFIYPV
Helix 5: AGSGVVHMAGAAAALAGVLLL
Helix 6: LPLATLGTFILWLGWFGF
Helix 7: IFVNTNAGAAGGVIAALIVA
Helix 8: ADLTMAL

Helix 1: NAFMMICTALVLFMTIPGIALFY
Helix 2: MLTQVAVTFALVCVLWVVY
Helix 3: YIHVAFQGSFACITVGLIVGALA
Helix 4: AVLIFVVVWLTLSYIPIAHMVWG
Helix 5: FAGGTVVHINAAVAGLVGAYLIG
Helix 6: PHNLPMVFTGTAILYFGWFG
Helix 7: IAALAFVNTVVATAGAILSWV
Helix 8: LLGACSGAIAGLVGITPA
Helix 9: VGGALLIGIVAGLAGLWGVTAL
Helix 10: VFGVHGVCGIIGCVMTGIFAA
Helix 11: VQLESIAITVVWSGVVAFIGY

Query ID: A0A801DSC9
Helix 1: NAFMMICTALVLFMTIPGIALFY
Helix 2: MLTQVAVTFALVCVLWVVY
Helix 3: YIHVAFQGSFACITVGLIVGALA
Helix 4: AVLIFVVVWLTLSYIPIAHMVWG
Helix 5: FAGGTVVHINAAVAGLVGAYLIG
Helix 6: PHNLPMVFTGTAILYFGWFG
Helix 7: IAALAFVNTVVATAGAILSWV
Helix 8: LLGACSGAIAGLVGITPA
Helix 9: VGGALLIGIVAGLAGLWGVTAL
Helix 10: VFGVHGVCGIVGCIMTGIFAA
Helix 11: VQLESIAITVVWSGVVAFIGY

Query ID: A0A2W0GQV0
Helix 1: ISTALVLFMSIPGIALFYGGLI
Helix 2: VAVTFALVCVLWVVYGYSLAFGTGNAFF
Helix 3: FYQYIHVAFQGSFACITVGLIVGALA
Helix 4: AVLIFVVVWLTLSYVPIAHMVWG
Helix 5: FAGGTVVHINAAVAGLVGAYLIG
Helix 6: PHNLPMVFTGTAILYFGWFG
Helix 7: IAALAFVNTVVATAGAILSWV
Helix 8: LLGACSGAIAGLVGITPA
Hel

Helix 1: AWMMTATVLVLLMILPGLALFY
Helix 2: TMTQIGATAALAMLVWVMW
Helix 3: YVFISFQMTFAAITAALIL
Helix 4: VMAFVPIWLTIVYFPIAHMVWAG
Helix 5: FAGGTVVHINAGVSGLVLAYLLG
Helix 6: PHSLTLTMVGTGLLWVGWFGF
Helix 7: AGLAMINTFVATAAAALTWM
Helix 8: ALGFCSGVIAGLVAVTPAA
Helix 9: FGAILLGILSSAVCYYFVA
Helix 10: LDAFGIHGIGGIVGAIGTGIVYQPF
Helix 11: FGVLVTIAWAGIGTLVAAYIV

Query ID: A0A7V8RCZ2
Helix 1: TWMLVSAVLVLMMSIPGLALFY
Helix 2: VLMQVLTIVSVAALVWVSWGY
Helix 3: LAFVVFQMTFACITPALIVGAFA
Helix 4: PLIIFTVLWLTLAYFPVAHMVWYWA
Helix 5: ALDFAGGTVVHINAGIAGLVGCLVIG
Helix 6: PHSLVMTMIGASLLWVGWFGF
Helix 7: TAVAFINTFVATAAATVAWAV
Helix 8: LGAATGAVAGLVAITPASGLA
Helix 9: ILLGFAASIVCFFFVTTV
Helix 10: VFGVHCVGGIIGAIGTGIVA
Helix 11: AVGITILWSGLVSAVLFYAL

Query ID: A0A7V8RAY5
Helix 1: AFMLICGALLMLFAVPGVLLM
Helix 2: GVFLHGTAIAALVSLLWVVV
Helix 3: AFVLFQIAGAMLAALLVAGAWA
Helix 4: WALAFGGMWSLIVYAPVAHWIWG
Helix 5: YAGGIALHFCAGISALVAALVI
Helix 6: ATHNPVMVLAGGALLFIGNFAL
Helix 7: ASSAIINAHVAACTSALVWLLI
Helix 8: TVSAMGAVNGLLAGLAAIAPAS
Helix 9: 

...

Helix 1: FFLIFAGALVFMMQAGFAMLCAGSV
Helix 2: NLLDACGGAIGFYTVGFGFAYG
Helix 3: YTNYAGFFFQFAFAATAATIVA
Helix 4: YLCYSLFLTGFVYPVVVRSVW
Helix 5: VDFAGSGVVHMTGGLTALIAAIVL
Helix 6: FPAHSVALQILGTFILWFGWYGF
Helix 7: ATAALCAVTTTMAAAAGCVSAMFT
Helix 8: TTYDLTMAMNGCLAGLVAVT
Helix 9: VTPWAAIIIGVVGGWVYIGM
Helix 10: FANGFWGVLATGLFANGGLMATA
Helix 11: GSLLICQLACLAWIIGWVTTIMTPFFILL

Query ID: B7FSA7
Helix 1: TFAGAIVFLMQAGFAMVCAGAV
Helix 2: NLLDACGASLAFFSIGYALG
Helix 3: YAFWLFQYAFSAASATIVAGTLA
Helix 4: AYLCYSLMLTGWVYPVILHSIW
Helix 5: VDFAGSGVVHVTGGITALFATMVL
Helix 6: ALQMLGTFILWFGFYGFNIGSALI
Helix 7: LAGVNTTLSASAAGIVALFSNLWY
Helix 8: LKYAMNGAICGLVAISGG
Helix 9: PWAAVVTGAVAGVIYLLG
Helix 10: AVDAIPVHLCGGAWGILAVGLFAA
Helix 11: VLFGIQLIGLMFIMGWVMIIMLPFFVWL

Query ID: B7FW68
Helix 1: WLMLFSGGLIFFMQTGFAMLCAGCV
Helix 2: NLLDACGAALGFFLLGYAFA
Helix 3: FWFFQFAFSATAVTIVAGTL
Helix 4: VAYLCYSIFLTGFVYPVAAHTIW
Helix 5: IDFAGSGVVHVTGGTTALVATYIL
Helix 6: FPGHSVALQLLGTFVLWFGW
Helix 7: VASRAAVNTSLSAASGAVSALMT
Helix 8: YSFNII

Helix 1: AWMLTASALVMVMTPGVAFFYAGL
Helix 2: MSFVSMALVSIQFWAFGYSAAFGSHGVF
Helix 3: IPHLLFAFFQTQFAMITPALL
Helix 4: TFLLFILLWTSLVYDPLAHWMW
Helix 5: FAGGTVIHISSGFGALAAALMVG
Helix 6: HNVPLVMIGVTLLWFGWFGF
Helix 7: AAIAAINTHLAASSGFLTWVGL
Helix 8: KFDPCGAASGAVAGLVAIT
Helix 9: VYPWASVIFGVVGAATGFSAI
Helix 10: LDSFAIHGCVGIMGGLLTGLFA
Helix 11: FVHQLVSQCVAAAYSFVVTMIILYLL

Query ID: H3G9J8
Helix 1: HVMIYIGFGFLMTFLRKY
Helix 2: LNFLIAVLALEWGIIAVTMAHQI
Helix 3: IPTMINGDFAAGAVLISFGAVLG
Helix 4: LVWMTFLEIIFYAFNEYIVL
Helix 5: MVIHSFGAFFGLALTIQLGAP
Helix 6: YHSDVFAMIGTLFLWMYW
Helix 7: MNTVLSIAASCASAFVASKL
Helix 8: IQNATLAGGVAMGTSCNLAI
Helix 9: ITVGLVVGIASVFGFSFVSPRL
Helix 10: ILNLHGMPGVVGGLAGAIITFSA
Helix 11: WYQLLAIVSSVGIGTISGFLVGFFL

Query ID: D2R355
Helix 1: AWMLTSAALVLFMTAPGLAMFY
Helix 2: VMMQCIFLMGLMTVIWALY
Helix 3: MLFQGMFFIITPALICGAFA
Helix 4: TMVVFSILWGTILYCPLCHMVW
Helix 5: FAGGTVVHISSGVSALVCALVIG
Helix 6: PHNLTYTTLGAAMLWVGWFGF
Helix 7: TSSAFAVTHFSAAAGTLSWA
Helix 8: VLGACSGAVAGLVCITPAA
Helix 9: MAAL

Helix 1: IAFIAICGILVSFMVPGVAFLY
Helix 2: LIWAVAASNAVVIFQWFFW
Helix 3: LLYSFYQMEFACVTVAILMGAL
Helix 4: VPAMIFAFIWMTIVYCPLACWAWA
Helix 5: GGGPVEIGSGVSGLAYSWVLG
Helix 6: PHNVSQVCMGTFMLWFGWLGF
Helix 7: AVLAIWNSMIAAAMAGLVWCLL
Helix 8: KVSMVGFCSGTIAGLVAAT
Helix 9: IPQWASLIMGIVVGVVANYAT
Helix 10: LDLYAEHAIGGIVGLLFNALF
Helix 11: LYIQFAYVCAVVGYSFVVTALLA

Query ID: A0A8H7P8U1
Helix 1: IAWVLCSTALVWIMIPGVGFFY
Helix 2: MIWMCMMTIAVVSFQGTRCSFSLAF
Helix 3: VPSIAFCVFQLMFAAITPMLAI
Helix 4: VMVFVFIWSTIVYDPIACWTWN
Helix 5: FILGGLDFAGGTPVHISSGTAALAI
Helix 6: NTTYVVLGTVFLWFGWFGF
Helix 7: AVQACIVTNLAASVGGLTWMLW
Helix 8: KWSTVGFCSGAIAGLVAI
Helix 9: FVGSPAAVLFGFMAGTVCNFA
Helix 10: CLDIFASHAIGGVVGNILTGLFA
Helix 11: GMSYSFVMTTIILWVMHYI

Query ID: A0A1X6N3Q5
Helix 1: IAWVLCSTALVWIMIPGVGFFY
Helix 2: MIWMCMMTIAVVSFQWFFW
Helix 3: VPSIAFCVFQLMFAAITPMLAI
Helix 4: VMVFVFIWSTIVYDPIACWTWN
Helix 5: FILGGLDFAGGTPVHISSGTAALAI
Helix 6: NTTYVVLGTVFLWFGWFGF
Helix 7: AVQACIVTNLAASVGGLTWML
Helix 8: VGFCSGAIAGLVAITPGSGFV
Helix 9:

query A0A9D9G322 has 12, using last 11 helices
Helix 1: LWLLIATILVIFMNAGFAMVE
Helix 2: ILAKNLFVFALAVTSYWFI
Helix 3: FLFQSAFAGTAATIVSGLVAE
Helix 4: FVVFAVVLTAFIYPIAGSW
Helix 5: IVHSVGAWAGLVGAMLLGPRIG
Helix 6: GHNMAIATLGALVLWIGWYGF
Helix 7: VPYVAVTTTLAAAAGAIGATIVSTL
Helix 8: LTMIINGILAGLVSITAG
Helix 9: LAGAWFAGLVGGIIVVFSVAAL
Helix 10: FSVHGVCGVWGTVVIGLWGTA
Helix 11: LVQALGAAAYAIWTLVTCWIAWSVI

Query ID: A0A9D9C0X8
query A0A9D9C0X8 has 12, using last 11 helices
Helix 1: LWLLIATILVIFMNAGFAMVE
Helix 2: ILAKNLFVFALAVTSYWFI
Helix 3: FLFQSAFAGTAATIVSGLVAE
Helix 4: FVVFAVVLTAFIYPIAGSW
Helix 5: IVHSVGAWAGLVGAMLLGPRIG
Helix 6: GHNMAIATLGALVLWIGWYGF
Helix 7: VPYVAVTTTLAAAAGAIGATIVSTL
Helix 8: LTMIINGILAGLVSITAG
Helix 9: LAGAWFAGLVGGIIVVFSVAAL
Helix 10: FSVHGVCGVWGTVVIGLWGTA
Helix 11: LVQALGAAAYAIWTLVTCWIAWSVI

Query ID: Q46HB1
query Q46HB1 has 12, using last 11 helices
Helix 1: LWLFIATILVIFMNAGFAMVE
Helix 2: ILAKNLFVFALAVTAYWVI
Helix 3: FLFQSAFAGTAATIVSGLVAE
Helix 4: FVVFSIVLTAFIYPIAGSWQWN
Helix 5:

Helix 1: TIQLTLTTYLVMIGAGQLLF
Helix 2: VLLGGGLAYVVASMGLAL
Helix 3: VFLGLRILQACGASACLVSTFATV
Helix 4: VIYGILGSMLAMVPAVGPLLGALV
Helix 5: WRAIFAFLGLGMIAASAAAW
Helix 6: FWLYTLCYAAGMGSFFVFFSIA
Helix 7: QLGFSLLFATVAIAMVFTARFM
Helix 8: VLRMGMGCLIAGAVLLAITEIWAL
Helix 9: FIAPMWLVGIGVATAVSVAPNGAL
Helix 10: AVYFCLGGVLLGSIGTLIISLL
Helix 11: WPVVVYCLTLATVVLGLSCV

Query ID: A0A264VQM8
Helix 1: NAFMMIATALVIFMILPGIALFY
Helix 2: LMAQTAVIFALVSVIWIVYGY
Helix 3: FIHVVFQGAFACLTVALVIGALG
Helix 4: AILIFTVIWTTFAYLPMAHMVWG
Helix 5: FAGGTVVHINAAVAGLVGAYLLG
Helix 6: PHNLPMVFMGTSILFIGWFG
Helix 7: IAALAFVNTIAAAAGAILSWT
Helix 8: LLGACSGCLAGLVGVTPAAG
Helix 9: GALIIGLLSGLAGLWGVVIL
Helix 10: VFGVHGVCGILGCLLTGVF
Helix 11: VGIQAISILVCVVWTAIVAYIAF

Query ID: A0A1Z1SZE8
Helix 1: ICSALVFFMTIPGIALFYGGLL
Helix 2: VMLIFSVVIILWIMFGYSLAFTA
Helix 3: YVHVVFQGSFAVITVALIVGALG
Helix 4: ALLIFTVIWFTFSYIPIAHMVW
Helix 5: LDFAGGTVVHINAAVAALVGAYLLG
Helix 6: PHNLPMVFTGTAVLYIGWFG
Helix 7: IAALAFLNTVVATAGAVLAWT
Helix 8: MLGSCSGCIAGLVAITPAA
H

In [8]:
len(matches)

11561

In [9]:
with open('lest.txt', 'w') as fp:
    for ite in lest:
        # write each item on a new line
        fp.write("%s\n" % ite)
    print('Done')

Done


In [10]:
with open('spoen.txt', 'w') as fp:
    for items in spoen:
        # write each item on a new line
        fp.write("%s\n" % items)
    print('Done')

Done


In [11]:
with open('matches.txt', 'w') as fp:
    for itemi in matches:
        # write each item on a new line
        fp.write("%s\n" % itemi)
    print('Done')

Done


In [12]:
strong = ' '.join([str(item) for item in lest])

In [13]:

stant = strong.replace(' ', '\n')


In [14]:
textfile = open('helices2.fasta', 'w')
textfile.write(stant)
textfile.close()

In [15]:
def countOccurrence(a):
  k = {}
  for j in a:
    if j in k:
      k[j] +=1
    else:
      k[j] =1
  return k


ths = countOccurrence(spoen)

In [16]:
ths = {key:val for key, val in ths.items() if val >= 11}

In [17]:
hip = []

for u in ths:
    hip.append(u)

In [18]:
with open('hip', 'w') as fp:
    for iteme in hip:
        # write each item on a new line
        fp.write("%s\n" % iteme)
    print('Done')

Done


In [19]:
len(hip)

1051

In [20]:
hip

['A0A011VQM7',
 'A0A063ZJU6',
 'A0A081EVR3',
 'A0A099I9L1',
 'A0A0D8IWY3',
 'A0A0E2HBL2',
 'A0A0F0CIE1',
 'A0A0G3WG46',
 'A0A0G9LEF6',
 'A0A0H5SG89',
 'A0A0J6ZRU4',
 'A0A0J9EMW5',
 'A0A0M2NHW4',
 'A0A0M6WCZ2',
 'A0A0M9AS13',
 'A0A0S2W112',
 'A0A0U5JBG5',
 'A0A0W7TMH6',
 'A0A136Q6K1',
 'A0A136WGT4',
 'A0A140DTE7',
 'A0A143Y517',
 'A0A143Y9S4',
 'A0A143YNF2',
 'A0A143YTA3',
 'A0A143YZV1',
 'A0A143Z212',
 'A0A143ZCA8',
 'A0A151AI36',
 'A0A169X118',
 'A0A173R3X4',
 'A0A173R662',
 'A0A173V441',
 'A0A173X4T2',
 'A0A173XV88',
 'A0A173XVK8',
 'A0A173Z4K3',
 'A0A173ZTK6',
 'A0A174C5U1',
 'A0A174D1D6',
 'A0A174H6C4',
 'A0A174JG28',
 'A0A174RYE8',
 'A0A174TIU3',
 'A0A174VZ55',
 'A0A174XKK3',
 'A0A174YTX7',
 'A0A174Z5H4',
 'A0A175A4T9',
 'A0A1B1YE78',
 'A0A1B1YLE9',
 'A0A1C0BVB4',
 'A0A1C5M0W8',
 'A0A1C5MDL8',
 'A0A1C5N7S1',
 'A0A1C5PFK7',
 'A0A1C5PIG4',
 'A0A1C5PQC5',
 'A0A1C5QAX9',
 'A0A1C5QDA2',
 'A0A1C5QHK8',
 'A0A1C5QX74',
 'A0A1C5R1D9',
 'A0A1C5R365',
 'A0A1C5R381',
 'A0A1C5RAN1',
 'A0A1C5S8

In [21]:
u = UniProt(verbose=False)

In [22]:
losht = []

for bats in hip:
    headers = u.search(bats, frmt = 'fasta')
#     print(headers)
    record = SeqIO.read(StringIO(headers), "fasta")
    print(record)
    losht.append(record)

ID: tr|A0A011VQM7|A0A011VQM7_RUMAL
Name: tr|A0A011VQM7|A0A011VQM7_RUMAL
Description: tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1
Number of features: 0
Seq('MDVQEIVQEVTQASDKTVFGVWFMIAVALVFFMQAGFAMVETGFTRAKNAGNII...PAT')
ID: tr|A0A063ZJU6|A0A063ZJU6_9EURY
Name: tr|A0A063ZJU6|A0A063ZJU6_9EURY
Description: tr|A0A063ZJU6|A0A063ZJU6_9EURY Ammonium transporter OS=Halostagnicola sp. A56 OX=1495067 GN=EL22_10090 PE=3 SV=1
Number of features: 0
Seq('MDATPLQVDPEVIVEGVNLMWVLIATFLIFFMHAGFAMLEAGQVRSKNVANQLT...DAV')
ID: tr|A0A081EVR3|A0A081EVR3_9EURY
Name: tr|A0A081EVR3|A0A081EVR3_9EURY
Description: tr|A0A081EVR3|A0A081EVR3_9EURY Ammonium transporter OS=Halorubrum saccharovorum OX=2248 GN=FK85_01485 PE=3 SV=1
Number of features: 0
Seq('MTAGVLASIDPSVLAEGVNLMWVAVVCFLIFFMHAGFAMLEAGQVRAKNVANQL...EAV')
ID: tr|A0A099I9L1|A0A099I9L1_CLOIN
Name: tr|A0A099I9L1|A0A099I9L1_CLOIN
Description: tr|A0A099I9L1|A0A099I9L1_CLOIN Adenylate cyclase OS=Cl

ID: tr|W7QGE0|W7QGE0_9ALTE
Name: tr|W7QGE0|W7QGE0_9ALTE
Description: tr|W7QGE0|W7QGE0_9ALTE Ammonium transporter OS=Catenovulum agarivorans DS-2 OX=1328313 GN=DS2_05440 PE=3 SV=1
Number of features: 0
Seq('MRAIIALLLLLPLTASAEGLDTGDTAWILTASALVLFMTLPGLSFFYAGLVRSK...YNL')
ID: tr|W7QG13|W7QG13_9ALTE
Name: tr|W7QG13|W7QG13_9ALTE
Description: tr|W7QG13|W7QG13_9ALTE Ammonium transporter OS=Catenovulum agarivorans DS-2 OX=1328313 GN=DS2_06106 PE=3 SV=1
Number of features: 0
Seq('MENNVFHLQYAMDTFYFLVCGALVMWMAAGFAMLEAGLVRAKNTTEILTKNIAL...SAK')
ID: tr|W7QGM4|W7QGM4_9ALTE
Name: tr|W7QGM4|W7QGM4_9ALTE
Description: tr|W7QGM4|W7QGM4_9ALTE Diguanylate cyclase/phosphodiesterase (GGDEF & EAL domains) with PAS/PAC sensor(S) OS=Catenovulum agarivorans DS-2 OX=1328313 GN=DS2_00080 PE=3 SV=1
Number of features: 0
Seq('MTIDILWVLFCAVLVLVMQGGFLFLESGLTRKKNAINVALKNATDFALTFLLWW...LVS')
ID: tr|C7QGX3|C7QGX3_CATAD
Name: tr|C7QGX3|C7QGX3_CATAD
Description: tr|C7QGX3|C7QGX3_CATAD Ammonium transporter OS=Catenulispora a

ID: tr|A0A2K3DQF8|A0A2K3DQF8_CHLRE
Name: tr|A0A2K3DQF8|A0A2K3DQF8_CHLRE
Description: tr|A0A2K3DQF8|A0A2K3DQF8_CHLRE Ammonium transporter OS=Chlamydomonas reinhardtii OX=3055 GN=CHLRE_06g293051v5 PE=3 SV=1
Number of features: 0
Seq('MDGTAKAEVALSLDVAFLLFSAYLVFGPMQLGFALLCAGAIRSKNSMNVLMKNI...IAQ')
ID: tr|Q6QH01|Q6QH01_CHLRE
Name: tr|Q6QH01|Q6QH01_CHLRE
Description: tr|Q6QH01|Q6QH01_CHLRE Ammonium transporter OS=Chlamydomonas reinhardtii OX=3055 GN=Amt1-4 PE=2 SV=1
Number of features: 0
Seq('MADEMDPMTACITALTDAAMTTAQATALCGQFAFAADNSDLSDRLDQTNQGLNT...LGL')
ID: tr|Q8RUT6|Q8RUT6_CHLRE
Name: tr|Q8RUT6|Q8RUT6_CHLRE
Description: tr|Q8RUT6|Q8RUT6_CHLRE Ammonium transporter OS=Chlamydomonas reinhardtii OX=3055 GN=Amt1;1 PE=2 SV=1
Number of features: 0
Seq('MSGDFGSEPLGSCSVETVTALLGYGLEQDSITALCQPEGGAGCTSTDNCMFQYL...GKA')
ID: tr|Q8LJU0|Q8LJU0_CHLRE
Name: tr|Q8LJU0|Q8LJU0_CHLRE
Description: tr|Q8LJU0|Q8LJU0_CHLRE Ammonium transporter OS=Chlamydomonas reinhardtii OX=3055 PE=2 SV=2
Number of features: 0
Seq

ID: tr|A0A251YFS0|A0A251YFS0_9MICO
Name: tr|A0A251YFS0|A0A251YFS0_9MICO
Description: tr|A0A251YFS0|A0A251YFS0_9MICO Ammonium transporter OS=Clavibacter michiganensis OX=28447 GN=nrgA PE=3 SV=1
Number of features: 0
Seq('MDQGNTAFLLIAAALVLLMTPGLAFFYGGLVKAKSVISMMMMSFGAMGLIGLLW...SGR')
ID: tr|A0A251XJT0|A0A251XJT0_CLAMM
Name: tr|A0A251XJT0|A0A251XJT0_CLAMM
Description: tr|A0A251XJT0|A0A251XJT0_CLAMM Ammonium transporter OS=Clavibacter michiganensis subsp. michiganensis OX=33013 GN=nrgA PE=3 SV=1
Number of features: 0
Seq('MDQGNTAFLLIAAALVLLMTPGLAFFYGGLVKAKSVISMMMMSFGAMGLIGLLW...SGR')
ID: tr|A0A399NT81|A0A399NT81_9MICO
Name: tr|A0A399NT81|A0A399NT81_9MICO
Description: tr|A0A399NT81|A0A399NT81_9MICO Ammonium transporter OS=Clavibacter michiganensis OX=28447 GN=amt PE=3 SV=1
Number of features: 0
Seq('MDQGNTAFLLIAAALVLLMTPGLAFFYGGLVKAKSVISMMMMSFGAMGLIGLLW...SGR')
ID: tr|A5CRL2|A5CRL2_CLAM3
Name: tr|A5CRL2|A5CRL2_CLAM3
Description: tr|A5CRL2|A5CRL2_CLAM3 Ammonium transporter OS=Clavibacter mic

ID: tr|A0A127PIQ9|A0A127PIQ9_9BURK
Name: tr|A0A127PIQ9|A0A127PIQ9_9BURK
Description: tr|A0A127PIQ9|A0A127PIQ9_9BURK Ammonium transporter OS=Collimonas fungivorans OX=158899 GN=amt PE=3 SV=1
Number of features: 0
Seq('MNIKSTFATILAACTLIAATSMLPASAADEPSASASASVAAASADAPAAAPAAV...YHD')
ID: tr|G0AFH4|G0AFH4_COLFT
Name: tr|G0AFH4|G0AFH4_COLFT
Description: tr|G0AFH4|G0AFH4_COLFT Ammonium transporter OS=Collimonas fungivorans (strain Ter331) OX=1005048 GN=amtB PE=3 SV=1
Number of features: 0
Seq('MPASAADEPSASASASVAAASADAPAAASAAVAAAPAASAPAAAAPAASAPAAA...YHD')
ID: tr|A0A1Y5E3E3|A0A1Y5E3E3_COLPS
Name: tr|A0A1Y5E3E3|A0A1Y5E3E3_COLPS
Description: tr|A0A1Y5E3E3|A0A1Y5E3E3_COLPS Ammonium transporter OS=Colwellia psychrerythraea OX=28229 GN=A9Q75_17915 PE=3 SV=1
Number of features: 0
Seq('METTMSKALILCLTLLSFFISGTAWADEGQLNGANTSWILTSTALVLLMTLPGV...YHL')
ID: tr|A0A099KL22|A0A099KL22_COLPS
Name: tr|A0A099KL22|A0A099KL22_COLPS
Description: tr|A0A099KL22|A0A099KL22_COLPS Ammonium transporter OS=Colwellia psychr

...

ID: tr|C0QSY0|C0QSY0_PERMH
Name: tr|C0QSY0|C0QSY0_PERMH
Description: tr|C0QSY0|C0QSY0_PERMH Ammonium transporter OS=Persephonella marina (strain DSM 14350 / EX-H1) OX=123214 GN=PERMA_2024 PE=3 SV=1
Number of features: 0
Seq('MNIRLLALFSSLIPAVSFAEEAKLDTGDTAWMIVATAFVVLMSIGGLTLFYGGM...FNL')
ID: tr|C0QTA4|C0QTA4_PERMH
Name: tr|C0QTA4|C0QTA4_PERMH
Description: tr|C0QTA4|C0QTA4_PERMH Ammonium transporter OS=Persephonella marina (strain DSM 14350 / EX-H1) OX=123214 GN=PERMA_0121 PE=3 SV=1
Number of features: 0
Seq('MSIVPADNVWILTATALVFLMSIPGLALFYSGLSKGKSMLNTIMMVMVAFCVVS...EIL')
ID: tr|A9BIZ7|A9BIZ7_PETMO
Name: tr|A9BIZ7|A9BIZ7_PETMO
Description: tr|A9BIZ7|A9BIZ7_PETMO Ammonium transporter OS=Petrotoga mobilis (strain DSM 10674 / SJ95) OX=403833 GN=Pmob_1796 PE=3 SV=1
Number of features: 0
Seq('MKRVWKSLLMMSVAFIVPTVLLADSITMEDVVSSIDTMWTLLAAFLVFFMQAGF...EGE')
ID: tr|A0A1B0ZRL8|A0A1B0ZRL8_9RHOB
Name: tr|A0A1B0ZRL8|A0A1B0ZRL8_9RHOB
Description: tr|A0A1B0ZRL8|A0A1B0ZRL8_9RHOB Ammonium transporter OS=P

ID: tr|A0A5C4RNH5|A0A5C4RNH5_PHOLU
Name: tr|A0A5C4RNH5|A0A5C4RNH5_PHOLU
Description: tr|A0A5C4RNH5|A0A5C4RNH5_PHOLU Ammonium transporter OS=Photorhabdus luminescens subsp. sonorensis OX=1173677 GN=amtB PE=3 SV=1
Number of features: 0
Seq('MKKQFSVAAGVAASCVPSLSQAAEGVADKADNAFMMICTVLVLFMTIPGIALFY...AYN')
ID: tr|A0A6L9JJM6|A0A6L9JJM6_PHOLM
Name: tr|A0A6L9JJM6|A0A6L9JJM6_PHOLM
Description: tr|A0A6L9JJM6|A0A6L9JJM6_PHOLM Ammonium transporter OS=Photorhabdus laumondii subsp. laumondii OX=141679 GN=amtB PE=3 SV=1
Number of features: 0
Seq('MKRQFSVAASVVASCVPSLSQAAEGVVDKADNAFMMICTVLVLFMTIPGIALFY...AYN')
ID: tr|Q7N0M6|Q7N0M6_PHOLL
Name: tr|Q7N0M6|Q7N0M6_PHOLL
Description: tr|Q7N0M6|Q7N0M6_PHOLL Ammonium transporter OS=Photorhabdus laumondii subsp. laumondii (strain DSM 15139 / CIP 105565 / TT01) OX=243265 GN=amtB PE=3 SV=1
Number of features: 0
Seq('MRHCNSLIEKIDMKRQFSVAASVVASCVPSLSQAAEGVVDKADNAFMMICTVLV...AYN')
ID: tr|C7BJG7|C7BJG7_PHOAA
Name: tr|C7BJG7|C7BJG7_PHOAA
Description: tr|C7BJG7|C7BJG7_P

ID: tr|A0A2K1RA77|A0A2K1RA77_POPTR
Name: tr|A0A2K1RA77|A0A2K1RA77_POPTR
Description: tr|A0A2K1RA77|A0A2K1RA77_POPTR Ammonium transporter OS=Populus trichocarpa OX=3694 GN=POPTR_T000200 PE=3 SV=1
Number of features: 0
Seq('MAAPPPNLVPVAYQGGSPSVPDWLNKGDNAWQMISATLVGLQSVPGLVILYGSI...QVM')
ID: tr|A0A2K1RA81|A0A2K1RA81_POPTR
Name: tr|A0A2K1RA81|A0A2K1RA81_POPTR
Description: tr|A0A2K1RA81|A0A2K1RA81_POPTR Ammonium transporter OS=Populus trichocarpa OX=3694 GN=POPTR_T000600 PE=3 SV=1
Number of features: 0
Seq('MAAPPPNPVPVAYRGGLVASVPDWLNKGDNAWQMISATLVGLQSVPGLVILYGS...QVM')
ID: tr|A0A2K1WV98|A0A2K1WV98_POPTR
Name: tr|A0A2K1WV98|A0A2K1WV98_POPTR
Description: tr|A0A2K1WV98|A0A2K1WV98_POPTR Ammonium transporter OS=Populus trichocarpa OX=3694 GN=POPTR_018G033500 PE=3 SV=1
Number of features: 0
Seq('MSNDTAFPPNLLPDEASPEWFNKADNAWQLTAATLVGLQSIPGLMILYGGGVKK...EKT')
ID: tr|A0A8H7U3I1|A0A8H7U3I1_9APHY
Name: tr|A0A8H7U3I1|A0A8H7U3I1_9APHY
Description: tr|A0A8H7U3I1|A0A8H7U3I1_9APHY Ammonium transporter OS=Po

ID: tr|Q31CR9|Q31CR9_PROM9
Name: tr|Q31CR9|Q31CR9_PROM9
Description: tr|Q31CR9|Q31CR9_PROM9 Ammonium transporter OS=Prochlorococcus marinus (strain MIT 9312) OX=74546 GN=PMT9312_0265 PE=3 SV=1
Number of features: 0
Seq('MTTALQTPQRRSRTRLQDASLVNGPMLLLRSIRGFSSNRSMLWLATVPLALFGL...SAK')
ID: tr|A0A0A2A5X1|A0A0A2A5X1_PROMR
Name: tr|A0A0A2A5X1|A0A0A2A5X1_PROMR
Description: tr|A0A0A2A5X1|A0A0A2A5X1_PROMR Ammonium transporter OS=Prochlorococcus marinus str. MIT 9302 OX=74545 GN=EU96_1970 PE=3 SV=1
Number of features: 0
Seq('MTTALQTPQRRSRSRLQDASLVNGPMLLLRSIRGFSSNRSMLWLATVPLALFGL...SAK')
ID: tr|A0A0M2PZN7|A0A0M2PZN7_PROHO
Name: tr|A0A0M2PZN7|A0A0M2PZN7_PROHO
Description: tr|A0A0M2PZN7|A0A0M2PZN7_PROHO Ammonium transporter OS=Prochlorothrix hollandica PCC 9006 = CALU 1027 OX=317619 GN=PROH_09485 PE=3 SV=1
Number of features: 0
Seq('MVHSLRQFLPRFQWAWVACVPLTALIVIGWGLAAHAQDAPEMDPMYPIFLLNNL...ISE')
ID: tr|A0A5M4AWD2|A0A5M4AWD2_9BACT
Name: tr|A0A5M4AWD2|A0A5M4AWD2_9BACT
Description: tr|A0A5M4AWD2|A0A5M4A

ID: tr|A0A927UCK5|A0A927UCK5_9FIRM
Name: tr|A0A927UCK5|A0A927UCK5_9FIRM
Description: tr|A0A927UCK5|A0A927UCK5_9FIRM Ammonium transporter OS=Pseudobutyrivibrio ruminis OX=46206 GN=amt PE=3 SV=1
Number of features: 0
Seq('MEEYSSMLFGVWFLIGAALVFWMQAGFAMVEAGFTRAKNTGNIIMKNLMDFCIG...DVE')
ID: tr|A0A1H7IQT2|A0A1H7IQT2_9FIRM
Name: tr|A0A1H7IQT2|A0A1H7IQT2_9FIRM
Description: tr|A0A1H7IQT2|A0A1H7IQT2_9FIRM Ammonium transporter, Amt family OS=Pseudobutyrivibrio ruminis OX=46206 GN=SAMN02910377_01423 PE=3 SV=1
Number of features: 0
Seq('MEAYSSMLFGVWFLIGAALVFWMQAGFAMVEAGFTRAKNTGNIIMKNLMDFCIG...DVE')
ID: tr|A0A285RSQ4|A0A285RSQ4_9FIRM
Name: tr|A0A285RSQ4|A0A285RSQ4_9FIRM
Description: tr|A0A285RSQ4|A0A285RSQ4_9FIRM Ammonium transporter, Amt family OS=Pseudobutyrivibrio ruminis DSM 9787 OX=1123011 GN=SAMN02910411_1303 PE=3 SV=1
Number of features: 0
Seq('MEAYSSMLFGVWFLIGAALVFWMQAGFAMVEAGFTRAKNTGNIIMKNLMDFCIG...DVE')
ID: tr|A0A2G3DSU6|A0A2G3DSU6_9FIRM
Name: tr|A0A2G3DSU6|A0A2G3DSU6_9FIRM
Description: tr

In [23]:
lousht = ','.join([str(item) for item in losht])

In [24]:
def Convert(string):
    li = list(string.split(","))
    return li

In [25]:
def Convart(strung):
    la = list(strung.split('\n'))
    return la
    

In [26]:
boxes = Convart(lousht)

In [27]:
nmes = []

for x in boxes:
    if 'Description' in x:
        nmes.append(x)
        
nmes

['Description: tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1',
 'Description: tr|A0A063ZJU6|A0A063ZJU6_9EURY Ammonium transporter OS=Halostagnicola sp. A56 OX=1495067 GN=EL22_10090 PE=3 SV=1',
 'Description: tr|A0A081EVR3|A0A081EVR3_9EURY Ammonium transporter OS=Halorubrum saccharovorum OX=2248 GN=FK85_01485 PE=3 SV=1',
 'Description: tr|A0A099I9L1|A0A099I9L1_CLOIN Adenylate cyclase OS=Clostridium innocuum OX=1522 GN=CIAN88_08185 PE=3 SV=1',
 'Description: tr|A0A0D8IWY3|A0A0D8IWY3_9FIRM Adenylate cyclase OS=Ruthenibacterium lactatiformans OX=1550024 GN=TQ39_16150 PE=3 SV=1',
 'Description: tr|A0A0E2HBL2|A0A0E2HBL2_9FIRM Ammonium transporter OS=[Clostridium] clostridioforme 90A8 OX=999408 GN=HMPREF1090_02007 PE=3 SV=1',
 'Description: tr|A0A0F0CIE1|A0A0F0CIE1_9CLOT Ammonium transporter NrgA OS=Clostridium sp. FS41 OX=1609975 GN=nrgA PE=3 SV=1',
 'Description: tr|A0A0G3WG46|A0A0G3WG46_9BACT Ammonium transporter OS=Endomicro

In [28]:
baxes = ', '.join([str(item) for item in nmes])

In [29]:
baxes = baxes.replace('Description: Description: ', '')

In [30]:
baxes = baxes.replace('Description: ', '')

In [31]:
baxes = Convert(baxes)


In [32]:
baxes



['tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1',
 ' tr|A0A063ZJU6|A0A063ZJU6_9EURY Ammonium transporter OS=Halostagnicola sp. A56 OX=1495067 GN=EL22_10090 PE=3 SV=1',
 ' tr|A0A081EVR3|A0A081EVR3_9EURY Ammonium transporter OS=Halorubrum saccharovorum OX=2248 GN=FK85_01485 PE=3 SV=1',
 ' tr|A0A099I9L1|A0A099I9L1_CLOIN Adenylate cyclase OS=Clostridium innocuum OX=1522 GN=CIAN88_08185 PE=3 SV=1',
 ' tr|A0A0D8IWY3|A0A0D8IWY3_9FIRM Adenylate cyclase OS=Ruthenibacterium lactatiformans OX=1550024 GN=TQ39_16150 PE=3 SV=1',
 ' tr|A0A0E2HBL2|A0A0E2HBL2_9FIRM Ammonium transporter OS=[Clostridium] clostridioforme 90A8 OX=999408 GN=HMPREF1090_02007 PE=3 SV=1',
 ' tr|A0A0F0CIE1|A0A0F0CIE1_9CLOT Ammonium transporter NrgA OS=Clostridium sp. FS41 OX=1609975 GN=nrgA PE=3 SV=1',
 ' tr|A0A0G3WG46|A0A0G3WG46_9BACT Ammonium transporter OS=Endomicrobium proavitum OX=1408281 GN=amtB PE=3 SV=1',
 ' tr|A0A0G9LEF6|A0A0G9LEF6_9CLOT Adenylate cyclase

In [92]:
!seqkit grep -r -v -n -p '.*Helix_10.*'  helices2.fasta > helix.fasta
!seqkit grep -r -v -n -p '.*Helix_11.*'  helix.fasta > helixone.fasta
# !seqkit grep -r -v -n -p '.*Helix_12.*'  helixone.fasta > helixgone.fasta
!seqkit grep -r -n -p '.*Helix_1.*'  helixone.fasta > helix1.fasta

In [93]:
!seqkit grep -r -n -p '.*Helix_2.*'  helices2.fasta > helix2.fasta
!seqkit grep -r -n -p '.*Helix_3.*'  helices2.fasta > helix3.fasta
!seqkit grep -r -n -p '.*Helix_4.*'  helices2.fasta > helix4.fasta
!seqkit grep -r -n -p '.*Helix_5.*'  helices2.fasta > helix5.fasta
!seqkit grep -r -n -p '.*Helix_6.*'  helices2.fasta > helix6.fasta
!seqkit grep -r -n -p '.*Helix_7.*'  helices2.fasta > helix7.fasta
!seqkit grep -r -n -p '.*Helix_8.*'  helices2.fasta > helix8.fasta
!seqkit grep -r -n -p '.*Helix_9.*'  helices2.fasta > helix9.fasta
!seqkit grep -r -n -p '.*Helix_10.*'  helices2.fasta > helix10.fasta
!seqkit grep -r -n -p '.*Helix_11.*'  helices2.fasta > helix11.fasta
# !seqkit grep -r -n -p '.*Helix_12.*'  helices.fasta > helix12.fasta

In [35]:
with open('baxes.txt', 'w') as fp:
    for itemq in baxes:
        # write each item on a new line
        fp.write("%s\n" % itemq)
    print('Done')

Done


In [60]:
filtere = []

for hops in hip:
    for ide in baxes:
        if hops in ide:
            juic = ide
            if juic[0] == ' ':
                juic = juic[1:]
            filtere.append(juic)

In [61]:
filtere

['tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1',
 'tr|A0A063ZJU6|A0A063ZJU6_9EURY Ammonium transporter OS=Halostagnicola sp. A56 OX=1495067 GN=EL22_10090 PE=3 SV=1',
 'tr|A0A081EVR3|A0A081EVR3_9EURY Ammonium transporter OS=Halorubrum saccharovorum OX=2248 GN=FK85_01485 PE=3 SV=1',
 'tr|A0A099I9L1|A0A099I9L1_CLOIN Adenylate cyclase OS=Clostridium innocuum OX=1522 GN=CIAN88_08185 PE=3 SV=1',
 'tr|A0A0D8IWY3|A0A0D8IWY3_9FIRM Adenylate cyclase OS=Ruthenibacterium lactatiformans OX=1550024 GN=TQ39_16150 PE=3 SV=1',
 'tr|A0A0E2HBL2|A0A0E2HBL2_9FIRM Ammonium transporter OS=[Clostridium] clostridioforme 90A8 OX=999408 GN=HMPREF1090_02007 PE=3 SV=1',
 'tr|A0A0F0CIE1|A0A0F0CIE1_9CLOT Ammonium transporter NrgA OS=Clostridium sp. FS41 OX=1609975 GN=nrgA PE=3 SV=1',
 'tr|A0A0G3WG46|A0A0G3WG46_9BACT Ammonium transporter OS=Endomicrobium proavitum OX=1408281 GN=amtB PE=3 SV=1',
 'tr|A0A0G9LEF6|A0A0G9LEF6_9CLOT Adenylate cyclase OS=Clos

In [39]:
with open('filtere.txt', 'w') as fp:
    for itema in filtere:
        # write each item on a new line
        fp.write("%s\n" % itema)
    print('Done')

Done


In [62]:
filtere11 = np.repeat(filtere, 11)

In [41]:
with open('filtere11.txt', 'w') as fp:
    for itemd in filtere:
        # write each item on a new line
        fp.write("%s\n" % itemd)
    print('Done')

Done


In [63]:
len(filtere11)

11561

In [64]:
df = pd.DataFrame(filtere11)

In [44]:

# opening the file in read mode 
my_file = open("filtere11.txt", "r") 
  
# reading the file 
data = my_file.read() 
  
# replacing end of line('/n') with ' ' and 
# splitting the text it further when '.' is seen. 
filtere = data.replace('\n', ' ').split(".") 
  
# printing the data 
print(filtere) 
my_file.close() 


['tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1

In [59]:
filtere

['tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1 tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cyclase OS=Ruminococcus albus SY3 OX=1341156 GN=RASY3_13175 PE=3 SV=1

In [65]:
df

,0
0,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...
1,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...
2,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...
3,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...
4,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...
...,...
11556,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...
11557,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...
11558,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...
11559,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...


In [55]:
len(hip)

1051

In [46]:
str_end = []


for boots in hip:
    result = u.search(boots, frmt="tsv", columns='ft_transmem')
#     print(result)
    transmems = result.split("\n")[1]
    transmems = re.findall("(?<=TRANSMEM )(?P<tmemstart>[0-9]*)..(?P<tmemend>[0-9]*)", transmems)
#     print(transmems)
    transmems = [(int(start), int(end)) for start, end in transmems]
#     print(transmems)
    starts = list(parse_ft_transmembrane(result))
    if len(starts) != 11:
        print(f"{boots} has {len(starts)} helices, will use last 11")
        starts = starts[-11:]
    for start, end in starts:
        print(f"{boots}/{start}-{end}'")
        backs = f'/{start}-{end}'
        str_end.append(backs)


A0A011VQM7/20-40'
A0A011VQM7/61-78'
A0A011VQM7/98-119'
A0A011VQM7/131-152'
A0A011VQM7/164-186'
A0A011VQM7/207-226'
A0A011VQM7/241-262'
A0A011VQM7/269-287'
A0A011VQM7/293-313'
A0A011VQM7/325-344'
A0A011VQM7/374-399'
A0A063ZJU6/20-39'
A0A063ZJU6/60-82'
A0A063ZJU6/102-125'
A0A063ZJU6/137-159'
A0A063ZJU6/179-200'
A0A063ZJU6/221-238'
A0A063ZJU6/265-286'
A0A063ZJU6/293-311'
A0A063ZJU6/317-337'
A0A063ZJU6/349-369'
A0A063ZJU6/381-402'
A0A081EVR3/20-37'
A0A081EVR3/58-79'
A0A081EVR3/99-119'
A0A081EVR3/126-149'
A0A081EVR3/161-182'
A0A081EVR3/203-220'
A0A081EVR3/240-265'
A0A081EVR3/277-293'
A0A081EVR3/299-317'
A0A081EVR3/329-352'
A0A081EVR3/364-385'
A0A099I9L1/6-28'
A0A099I9L1/49-71'
A0A099I9L1/91-112'
A0A099I9L1/124-145'
A0A099I9L1/157-178'
A0A099I9L1/199-216'
A0A099I9L1/236-257'
A0A099I9L1/264-281'
A0A099I9L1/287-308'
A0A099I9L1/320-342'
A0A099I9L1/362-382'
A0A0D8IWY3/6-28'
A0A0D8IWY3/49-67'
A0A0D8IWY3/87-108'
A0A0D8IWY3/120-141'
A0A0D8IWY3/153-174'
A0A0D8IWY3/195-212'
A0A0D8IWY3/232-253'
A0A0D8

A0A516N903/6-29'
A0A516N903/41-61'
A0A516N903/108-127'
A0A516N903/134-155'
A0A516N903/186-208'
A0A516N903/220-240'
A0A516N903/252-272'
A0A516N903/284-301'
A0A516N903/307-326'
A0A516N903/338-365'
A0A516N903/371-396'
B3PIH2/36-58'
B3PIH2/70-93'
B3PIH2/127-146'
B3PIH2/153-174'
B3PIH2/186-209'
B3PIH2/229-250'
B3PIH2/256-278'
B3PIH2/285-305'
B3PIH2/311-329'
B3PIH2/341-364'
B3PIH2/384-404'
B3PF77/15-35'
B3PF77/47-69'
B3PF77/111-130'
B3PF77/135-152'
B3PF77/177-199'
B3PF77/211-233'
B3PF77/253-275'
B3PF77/282-301'
B3PF77/307-328'
B3PF77/340-357'
B3PF77/363-388'
A0RUZ2/72-93'
A0RUZ2/100-119'
A0RUZ2/191-213'
A0RUZ2/220-241'
A0RUZ2/261-287'
A0RUZ2/299-319'
A0RUZ2/331-355'
A0RUZ2/362-380'
A0RUZ2/386-405'
A0RUZ2/417-436'
A0RUZ2/463-483'
A0RXK8/75-96'
A0RXK8/103-122'
A0RXK8/195-217'
A0RXK8/224-245'
A0RXK8/265-291'
A0RXK8/303-322'
A0RXK8/334-354'
A0RXK8/366-384'
A0RXK8/390-409'
A0RXK8/421-445'
A0RXK8/465-486'
U7VB41/12-35'
U7VB41/47-75'
U7VB41/95-119'
U7VB41/131-153'
U7VB41/165-186'
U7VB41/198-217'
U7

A0A251Z278/6-27'
A0A251Z278/39-60'
A0A251Z278/110-132'
A0A251Z278/139-160'
A0A251Z278/180-205'
A0A251Z278/217-236'
A0A251Z278/248-270'
A0A251Z278/282-300'
A0A251Z278/306-324'
A0A251Z278/336-362'
A0A251Z278/374-394'
A0A251YFS0/6-27'
A0A251YFS0/39-60'
A0A251YFS0/110-132'
A0A251YFS0/139-160'
A0A251YFS0/180-205'
A0A251YFS0/217-236'
A0A251YFS0/248-270'
A0A251YFS0/282-300'
A0A251YFS0/306-324'
A0A251YFS0/336-362'
A0A251YFS0/374-394'
A0A251XJT0/6-27'
A0A251XJT0/39-60'
A0A251XJT0/110-132'
A0A251XJT0/139-160'
A0A251XJT0/180-205'
A0A251XJT0/217-236'
A0A251XJT0/248-270'
A0A251XJT0/282-300'
A0A251XJT0/306-324'
A0A251XJT0/336-362'
A0A251XJT0/374-394'
A0A399NT81/6-27'
A0A399NT81/39-60'
A0A399NT81/110-132'
A0A399NT81/139-160'
A0A399NT81/180-205'
A0A399NT81/217-236'
A0A399NT81/248-270'
A0A399NT81/282-300'
A0A399NT81/306-324'
A0A399NT81/336-362'
A0A399NT81/374-394'
A5CRL2/6-27'
A5CRL2/39-60'
A5CRL2/110-132'
A5CRL2/139-160'
A5CRL2/180-205'
A5CRL2/217-236'
A5CRL2/248-270'
A5CRL2/282-300'
A5CRL2/306-324'
A

A0A076PL76/52-74'
A0A076PL76/86-106'
A0A076PL76/143-161'
A0A076PL76/168-186'
A0A076PL76/206-228'
A0A076PL76/240-259'
A0A076PL76/271-292'
A0A076PL76/304-324'
A0A076PL76/330-347'
A0A076PL76/359-377'
A0A076PL76/397-418'
A0A5A7MAJ9/52-74'
A0A5A7MAJ9/86-106'
A0A5A7MAJ9/143-161'
A0A5A7MAJ9/168-186'
A0A5A7MAJ9/206-228'
A0A5A7MAJ9/240-260'
A0A5A7MAJ9/272-292'
A0A5A7MAJ9/304-324'
A0A5A7MAJ9/330-347'
A0A5A7MAJ9/359-382'
A0A5A7MAJ9/402-421'
A0A096FEE6/52-74'
A0A096FEE6/86-106'
A0A096FEE6/143-161'
A0A096FEE6/168-186'
A0A096FEE6/206-228'
A0A096FEE6/240-259'
A0A096FEE6/271-292'
A0A096FEE6/304-324'
A0A096FEE6/330-347'
A0A096FEE6/359-377'
A0A096FEE6/397-418'
A0A8B4RWH8/52-74'
A0A8B4RWH8/86-106'
A0A8B4RWH8/143-161'
A0A8B4RWH8/168-186'
A0A8B4RWH8/206-228'
A0A8B4RWH8/240-259'
A0A8B4RWH8/271-292'
A0A8B4RWH8/304-324'
A0A8B4RWH8/330-347'
A0A8B4RWH8/359-377'
A0A8B4RWH8/397-418'
D0IWD0/52-74'
D0IWD0/86-106'
D0IWD0/143-161'
D0IWD0/168-186'
D0IWD0/206-228'
D0IWD0/240-260'
D0IWD0/272-292'
D0IWD0/304-324'
D0IWD0/

K8CKZ7/32-58'
K8CKZ7/70-97'
K8CKZ7/117-142'
K8CKZ7/149-171'
K8CKZ7/183-205'
K8CKZ7/217-236'
K8CKZ7/248-268'
K8CKZ7/280-302'
K8CKZ7/308-324'
K8CKZ7/336-355'
K8CKZ7/375-396'
K8C851/32-58'
K8C851/70-97'
K8C851/117-142'
K8C851/149-171'
K8C851/183-205'
K8C851/217-236'
K8C851/248-268'
K8C851/280-302'
K8C851/308-324'
K8C851/336-355'
K8C851/375-396'
A0A3Q9CHX8/32-58'
A0A3Q9CHX8/70-97'
A0A3Q9CHX8/117-142'
A0A3Q9CHX8/149-171'
A0A3Q9CHX8/183-205'
A0A3Q9CHX8/217-236'
A0A3Q9CHX8/248-268'
A0A3Q9CHX8/280-302'
A0A3Q9CHX8/308-324'
A0A3Q9CHX8/336-355'
A0A3Q9CHX8/375-396'
Q560M9/42-63'
Q560M9/75-98'
Q560M9/133-153'
Q560M9/160-183'
Q560M9/195-217'
Q560M9/229-253'
Q560M9/265-285'
Q560M9/297-315'
Q560M9/321-339'
Q560M9/351-370'
Q560M9/399-420'
F5HBK8/30-51'
F5HBK8/63-81'
F5HBK8/117-138'
F5HBK8/150-171'
F5HBK8/177-203'
F5HBK8/223-240'
F5HBK8/252-273'
F5HBK8/280-297'
F5HBK8/303-324'
F5HBK8/336-357'
F5HBK8/392-410'
Q5KPM5/42-63'
Q5KPM5/75-98'
Q5KPM5/133-153'
Q5KPM5/160-183'
Q5KPM5/195-217'
Q5KPM5/229-253'
Q5KP

...

A0A8T8BI10/86-107'
A0A8T8BI10/119-137'
A0A8T8BI10/179-201'
A0A8T8BI10/208-231'
A0A8T8BI10/261-283'
A0A8T8BI10/295-315'
A0A8T8BI10/327-347'
A0A8T8BI10/359-379'
A0A8T8BI10/385-402'
A0A8T8BI10/414-434'
A0A8T8BI10/460-482'
A0A378YY10/12-32'
A0A378YY10/44-66'
A0A378YY10/86-107'
A0A378YY10/119-137'
A0A378YY10/157-180'
A0A378YY10/201-221'
A0A378YY10/227-249'
A0A378YY10/256-272'
A0A378YY10/278-296'
A0A378YY10/308-329'
A0A378YY10/341-363'
A0A8T8BK87/12-32'
A0A8T8BK87/44-66'
A0A8T8BK87/86-107'
A0A8T8BK87/119-137'
A0A8T8BK87/157-180'
A0A8T8BK87/201-221'
A0A8T8BK87/227-249'
A0A8T8BK87/256-274'
A0A8T8BK87/280-298'
A0A8T8BK87/310-329'
A0A8T8BK87/341-363'
A0A379A0S9/52-73'
A0A379A0S9/94-113'
A0A379A0S9/147-166'
A0A379A0S9/173-195'
A0A379A0S9/207-228'
A0A379A0S9/249-266'
A0A379A0S9/286-305'
A0A379A0S9/312-330'
A0A379A0S9/336-358'
A0A379A0S9/370-387'
A0A379A0S9/393-418'
A0A0H3KTD3/43-63'
A0A0H3KTD3/70-90'
A0A0H3KTD3/126-148'
A0A0H3KTD3/155-177'
A0A0H3KTD3/189-211'
A0A0H3KTD3/223-242'
A0A0H3KTD3/254-273

A0A0J6AS72/32-54'
A0A0J6AS72/66-84'
A0A0J6AS72/123-142'
A0A0J6AS72/149-171'
A0A0J6AS72/183-205'
A0A0J6AS72/217-237'
A0A0J6AS72/249-268'
A0A0J6AS72/280-299'
A0A0J6AS72/305-324'
A0A0J6AS72/336-354'
A0A0J6AS72/374-396'
A0A086F0C2/32-54'
A0A086F0C2/66-84'
A0A086F0C2/123-142'
A0A086F0C2/149-171'
A0A086F0C2/183-205'
A0A086F0C2/217-237'
A0A086F0C2/243-268'
A0A086F0C2/280-299'
A0A086F0C2/305-324'
A0A086F0C2/336-354'
A0A086F0C2/374-396'
A0A433P2R5/32-54'
A0A433P2R5/66-84'
A0A433P2R5/123-142'
A0A433P2R5/149-171'
A0A433P2R5/183-205'
A0A433P2R5/217-237'
A0A433P2R5/243-268'
A0A433P2R5/280-299'
A0A433P2R5/305-324'
A0A433P2R5/336-354'
A0A433P2R5/374-396'
A0A7V8TJ53/32-54'
A0A7V8TJ53/66-84'
A0A7V8TJ53/123-142'
A0A7V8TJ53/149-171'
A0A7V8TJ53/183-205'
A0A7V8TJ53/217-237'
A0A7V8TJ53/243-268'
A0A7V8TJ53/280-299'
A0A7V8TJ53/305-324'
A0A7V8TJ53/336-354'
A0A7V8TJ53/374-396'
A0A221TCD6/32-54'
A0A221TCD6/66-84'
A0A221TCD6/123-142'
A0A221TCD6/149-171'
A0A221TCD6/183-205'
A0A221TCD6/217-237'
A0A221TCD6/243-268'


A0A5C4RNH5/32-58'
A0A5C4RNH5/70-97'
A0A5C4RNH5/117-138'
A0A5C4RNH5/145-163'
A0A5C4RNH5/183-205'
A0A5C4RNH5/217-236'
A0A5C4RNH5/248-268'
A0A5C4RNH5/280-299'
A0A5C4RNH5/305-324'
A0A5C4RNH5/336-356'
A0A5C4RNH5/376-396'
A0A6L9JJM6/32-54'
A0A6L9JJM6/66-86'
A0A6L9JJM6/117-138'
A0A6L9JJM6/145-163'
A0A6L9JJM6/183-205'
A0A6L9JJM6/217-236'
A0A6L9JJM6/248-268'
A0A6L9JJM6/280-299'
A0A6L9JJM6/305-324'
A0A6L9JJM6/336-356'
A0A6L9JJM6/376-396'
Q7N0M6/44-66'
Q7N0M6/78-98'
Q7N0M6/129-150'
Q7N0M6/157-175'
Q7N0M6/195-217'
Q7N0M6/229-248'
Q7N0M6/260-280'
Q7N0M6/292-311'
Q7N0M6/317-336'
Q7N0M6/348-368'
Q7N0M6/388-408'
C7BJG7/32-54'
C7BJG7/66-86'
C7BJG7/117-138'
C7BJG7/145-163'
C7BJG7/183-205'
C7BJG7/217-236'
C7BJG7/248-268'
C7BJG7/280-299'
C7BJG7/305-324'
C7BJG7/336-356'
C7BJG7/376-396'
A0A4R4K1P9/32-54'
A0A4R4K1P9/66-86'
A0A4R4K1P9/117-138'
A0A4R4K1P9/145-163'
A0A4R4K1P9/183-205'
A0A4R4K1P9/217-236'
A0A4R4K1P9/248-268'
A0A4R4K1P9/280-299'
A0A4R4K1P9/305-324'
A0A4R4K1P9/336-356'
A0A4R4K1P9/376-396'
A0A1G5PW

A0A8I1X2A6 has 12 helices, if 12, will use last 11, if other won't include
A0A8I1X2A6/78-98'
A0A8I1X2A6/110-128'
A0A8I1X2A6/170-190'
A0A8I1X2A6/197-215'
A0A8I1X2A6/235-256'
A0A8I1X2A6/268-288'
A0A8I1X2A6/300-324'
A0A8I1X2A6/331-348'
A0A8I1X2A6/354-375'
A0A8I1X2A6/387-407'
A0A8I1X2A6/427-451'
A0A0A2BBY0/39-58'
A0A0A2BBY0/70-88'
A0A0A2BBY0/130-150'
A0A0A2BBY0/157-175'
A0A0A2BBY0/195-216'
A0A0A2BBY0/228-248'
A0A0A2BBY0/260-284'
A0A0A2BBY0/291-308'
A0A0A2BBY0/314-335'
A0A0A2BBY0/347-367'
A0A0A2BBY0/387-411'
A0A0A1ZMZ9/39-58'
A0A0A1ZMZ9/70-88'
A0A0A1ZMZ9/130-150'
A0A0A1ZMZ9/157-175'
A0A0A1ZMZ9/195-215'
A0A0A1ZMZ9/235-252'
A0A0A1ZMZ9/264-285'
A0A0A1ZMZ9/297-317'
A0A0A1ZMZ9/323-341'
A0A0A1ZMZ9/348-370'
A0A0A1ZMZ9/390-415'
Q7VDS3 has 12 helices, if 12, will use last 11, if other won't include
Q7VDS3/83-103'
Q7VDS3/115-133'
Q7VDS3/184-203'
Q7VDS3/208-226'
Q7VDS3/246-266'
Q7VDS3/286-303'
Q7VDS3/315-336'
Q7VDS3/348-368'
Q7VDS3/374-390'
Q7VDS3/402-421'
Q7VDS3/441-467'
Q7V4T8 has 12 helices, if 12,

A0A1S5XW08/50-69'
A0A1S5XW08/81-98'
A0A1S5XW08/104-127'
A0A1S5XW08/139-162'
A0A1S5XW08/168-187'
A0A1S5XW08/215-236'
A0A1S5XW08/248-269'
A0A1S5XW08/281-304'
A0A1S5XW08/310-333'
A0A1S5XW08/345-366'
A0A1S5XW08/372-391'
A0A264VQM8/33-55'
A0A264VQM8/67-87'
A0A264VQM8/121-143'
A0A264VQM8/150-172'
A0A264VQM8/184-206'
A0A264VQM8/218-237'
A0A264VQM8/249-269'
A0A264VQM8/281-300'
A0A264VQM8/306-325'
A0A264VQM8/337-355'
A0A264VQM8/375-397'
A0A1Z1SZE8/40-61'
A0A1Z1SZE8/73-95'
A0A1Z1SZE8/123-145'
A0A1Z1SZE8/152-173'
A0A1Z1SZE8/185-209'
A0A1Z1SZE8/221-240'
A0A1Z1SZE8/252-272'
A0A1Z1SZE8/284-302'
A0A1Z1SZE8/308-328'
A0A1Z1SZE8/340-366'
A0A1Z1SZE8/378-400'
A0A1S1HRJ0/32-54'
A0A1S1HRJ0/66-86'
A0A1S1HRJ0/123-142'
A0A1S1HRJ0/149-171'
A0A1S1HRJ0/183-205'
A0A1S1HRJ0/217-236'
A0A1S1HRJ0/248-268'
A0A1S1HRJ0/280-298'
A0A1S1HRJ0/304-324'
A0A1S1HRJ0/336-358'
A0A1S1HRJ0/378-399'
A0A379GJ68/32-56'
A0A379GJ68/63-84'
A0A379GJ68/120-142'
A0A379GJ68/149-171'
A0A379GJ68/183-205'
A0A379GJ68/217-237'
A0A379GJ68/249-268'


In [47]:
with open('str_end.txt', 'w') as fp:
    for itemt in str_end:
        # write each item on a new line
        fp.write("%s\n" % itemt)
    print('Done')

Done


In [48]:

# # opening the file in read mode 
# my_file = open("str_end.txt", "r") 
  
# # reading the file 
# data = my_file.read() 
  
# # replacing end of line('/n') with ' ' and 
# # splitting the text it further when '.' is seen. 
# str_end = data.replace('\n', ' ').split(".") 
  
# # printing the data 
# print(str_end) 
# my_file.close() 


In [70]:
df

,0,numbers
0,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/20-40
1,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/61-78
2,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/98-119
3,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/131-152
4,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/164-186
...,...,...
11556,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/239-262
11557,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/269-287
11558,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/293-313
11559,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/325-344


In [66]:
len(str_end)

11561

In [67]:
df['numbers'] = str_end

In [72]:
df[['split', 'splat']] = df[0].str.split(pat=None, n=1, expand = True)

In [73]:
df['header'] = df['split'] + df['numbers'] + ' ' + df['splat']

In [74]:
df.to_csv('df.csv')

In [75]:
df2 = pd.read_csv('df.csv')

In [76]:
df2

,Unnamed: 0,0,numbers,split,splat,header
0,0,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/20-40,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/20-40 Adenylate...
1,1,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/61-78,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/61-78 Adenylate...
2,2,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/98-119,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/98-119 Adenylat...
3,3,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/131-152,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/131-152 Adenyla...
4,4,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/164-186,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/164-186 Adenyla...
...,...,...,...,...,...,...
11556,11556,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/239-262,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/239-262 Adenylate cycla...
11557,11557,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/269-287,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/269-287 Adenylate cycla...
11558,11558,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/293-313,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/293-313 Adenylate cycla...
11559,11559,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/325-344,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/325-344 Adenylate cycla...


In [77]:
ids = []

for i in df2['split']:
    ids.append(i.split('|')[1])

In [78]:
df2['ids'] = ids

In [79]:
df2

,Unnamed: 0,0,numbers,split,splat,header,ids
0,0,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/20-40,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/20-40 Adenylate...,A0A011VQM7
1,1,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/61-78,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/61-78 Adenylate...,A0A011VQM7
2,2,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/98-119,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/98-119 Adenylat...,A0A011VQM7
3,3,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/131-152,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/131-152 Adenyla...,A0A011VQM7
4,4,tr|A0A011VQM7|A0A011VQM7_RUMAL Adenylate cycla...,/164-186,tr|A0A011VQM7|A0A011VQM7_RUMAL,Adenylate cyclase OS=Ruminococcus albus SY3 OX...,tr|A0A011VQM7|A0A011VQM7_RUMAL/164-186 Adenyla...,A0A011VQM7
...,...,...,...,...,...,...,...
11556,11556,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/239-262,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/239-262 Adenylate cycla...,W7UD62
11557,11557,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/269-287,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/269-287 Adenylate cycla...,W7UD62
11558,11558,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/293-313,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/293-313 Adenylate cycla...,W7UD62
11559,11559,tr|W7UD62|W7UD62_RUMFL Adenylate cyclase OS=Ru...,/325-344,tr|W7UD62|W7UD62_RUMFL,Adenylate cyclase OS=Ruminococcus flavefaciens...,tr|W7UD62|W7UD62_RUMFL/325-344 Adenylate cycla...,W7UD62


In [87]:
df2['combine'] = df2['ids'] + '_' + df2['helices']

KeyError: 'helices'

In [ ]:
df2

In [81]:
headers = []

for itims in df['header']:
    headers.append(itims)

In [82]:
headers = df['header'].to_list()

In [83]:
len(headers)

11561

In [85]:
with open('headers.txt', 'w') as fp:
    for itemy in headers:
        # write each item on a new line
        fp.write("%s\n" % itemy)
    print('Done')

Done


In [88]:
this_file = open("matches.txt", "r") 
  
# reading the file 
date = this_file.read() 
  
# replacing end splitting the text  
# when newline ('\n') is seen. 
data_list = date.split("\n") 
print(data_list) 
this_file.close() 

['A0A011VQM7_Helix_1', 'A0A011VQM7_Helix_2', 'A0A011VQM7_Helix_3', 'A0A011VQM7_Helix_4', 'A0A011VQM7_Helix_5', 'A0A011VQM7_Helix_6', 'A0A011VQM7_Helix_7', 'A0A011VQM7_Helix_8', 'A0A011VQM7_Helix_9', 'A0A011VQM7_Helix_10', 'A0A011VQM7_Helix_11', 'A0A063ZJU6_Helix_1', 'A0A063ZJU6_Helix_2', 'A0A063ZJU6_Helix_3', 'A0A063ZJU6_Helix_4', 'A0A063ZJU6_Helix_5', 'A0A063ZJU6_Helix_6', 'A0A063ZJU6_Helix_7', 'A0A063ZJU6_Helix_8', 'A0A063ZJU6_Helix_9', 'A0A063ZJU6_Helix_10', 'A0A063ZJU6_Helix_11', 'A0A081EVR3_Helix_1', 'A0A081EVR3_Helix_2', 'A0A081EVR3_Helix_3', 'A0A081EVR3_Helix_4', 'A0A081EVR3_Helix_5', 'A0A081EVR3_Helix_6', 'A0A081EVR3_Helix_7', 'A0A081EVR3_Helix_8', 'A0A081EVR3_Helix_9', 'A0A081EVR3_Helix_10', 'A0A081EVR3_Helix_11', 'A0A099I9L1_Helix_1', 'A0A099I9L1_Helix_2', 'A0A099I9L1_Helix_3', 'A0A099I9L1_Helix_4', 'A0A099I9L1_Helix_5', 'A0A099I9L1_Helix_6', 'A0A099I9L1_Helix_7', 'A0A099I9L1_Helix_8', 'A0A099I9L1_Helix_9', 'A0A099I9L1_Helix_10', 'A0A099I9L1_Helix_11', 'A0A0D8IWY3_Helix_1', '

In [86]:
rees = dict(zip(combine, headers))

NameError: name 'combine' is not defined

In [ ]:
len(rees)

In [ ]:
new_dict = {k: v for k, v in zip(ods, headers)}

In [ ]:
len(new_dict)
new_dict

In [ ]:
def list_duplicates(seq):
  seen = set()
  seen_add = seen.add
  # adds all elements it doesn't know yet to seen and all other to seen_twice
  seen_twice = set( x for x in seq if x in seen or seen_add(x) )
  # turn the set into a list (as requested)
  return list( seen_twice )

In [ ]:
len(data_list)

In [ ]:
soon = list_duplicates(data_list)
nondup = []
dup = []

for fits in data_list:
    if fits in soon:
        dup.append(fits)
    else:
        nondup.append(fits)
        

In [ ]:
dup


In [ ]:
sopn = list_duplicates(soon)
undup = []
dupu = []

for thuts in data_into_list:
    if thuts in sopn:
        dupu.append(thuts)
    else:
        undup.append(thuts)

In [ ]:
len(nondup)

In [89]:
rees = {}
for key in matches:
    for value in headers:
        rees[key] = value
        headers.remove(value)
        break

In [90]:
len(rees)

11561

In [ ]:
# !seqkit rmdup -n helix1.fasta > helixrmdup1.fasta




In [91]:
with open("rees_dict.txt", 'w') as f:  
    for key, value in rees.items():  
        f.write('%s:%s\n' % (key, value))

In [94]:
with open('helix1.fasta') as original, open('helix1n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [95]:
with open('helix2.fasta') as original, open('helix2n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [96]:
with open('helix3.fasta') as original, open('helix3n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [97]:
with open('helix4.fasta') as original, open('helix4n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [98]:
with open('helix5.fasta') as original, open('helix5n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [99]:
with open('helix6.fasta') as original, open('helix6n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [100]:
with open('helix7.fasta') as original, open('helix7n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [101]:
with open('helix8.fasta') as original, open('helix8n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [102]:
with open('helix9.fasta') as original, open('helix9n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [103]:
with open('helix10.fasta') as original, open('helix10n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [104]:
with open('helix11.fasta') as original, open('helix11n2.fasta', 'w') as corrected:
    for seq_record in SeqIO.parse(original, 'fasta'):
        if seq_record.id in rees:
            seq_record.id = rees[seq_record.id]
        SeqIO.write(seq_record, corrected, 'fasta')

In [ ]:
loshtit = []

headers = u.search("B2I573", frmt = 'fasta')
#     print(headers)
record = SeqIO.read(StringIO(headers), "fasta")
print(record)
loshtit.append(record)